In [22]:
import pennylane as qml
from pennylane import numpy as np

# 1. Setup - On définit bien wires comme une liste [0]
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def circuit(weights, inputs):
    # CORRECTION : wires=[0] au lieu de wires=0
    qml.AngleEmbedding(inputs, wires=[0]) 
    qml.RY(weights[0], wires=[0])
    return qml.expval(qml.PauliZ(0))

# 2. Données (On s'assure que X est bien formaté pour AngleEmbedding)
# Définission des signaux d'entrée (0.1 et 0.2 pour le bleu, 3.0 et 3.1 pour le rouge)
X = np.array([[0.1], [0.2], [3.0], [3.1]], requires_grad=False)
# Imposition de la normalisation associative : 1.0 pour les signaux bleus, -1.0 pour les rouges
Y = np.array([1.0, 1.0, -1.0, -1.0], requires_grad=False)

# 3. Fonction de coût
def cost(weights):
    # On calcule la prédiction pour chaque entrée
    predictions = [circuit(weights, x) for x in X]
    return np.mean((np.array(predictions) - Y)**2)

# 4. Optimisation
opt = qml.GradientDescentOptimizer(stepsize=0.4)
weights = np.array([0.1], requires_grad=True) # Un petit décalage initial

print("Début de la normalisation associative...")
for i in range(61):
    weights = opt.step(cost, weights)
    if i % 10 == 0:
        print(f"Étape {i:02d} - Erreur : {cost(weights):.4f}")

print(f"\nPoids finaux (ton filtre instinctif) : {weights}")

# TEST FINAL : 
# Voyons si un signal inconnu (ex: 0.15) est bien "normalisé" vers le bleu (1.0): 0.9872
test_signal = np.array([0.15])
prediction = circuit(weights, test_signal)
print(f"Test signal 0.15 -> Qualia : {prediction:.4f} (Proche de 1 = Bleu)")

Début de la normalisation associative...
Étape 00 - Erreur : 0.0002
Étape 10 - Erreur : 0.0002
Étape 20 - Erreur : 0.0002
Étape 30 - Erreur : 0.0002
Étape 40 - Erreur : 0.0002
Étape 50 - Erreur : 0.0002
Étape 60 - Erreur : 0.0002

Poids finaux (ton filtre instinctif) : [0.05639818]
Test signal 0.15 -> Qualia : 0.9872 (Proche de 1 = Bleu)


In [25]:
print("Début de la normalisation associative...")
for i in range(61):
    weights = opt.step(cost, weights)
    if i % 10 == 0:
        print(f"Étape {i:02d} - Erreur : {cost(weights):.4f}")

print(f"\nPoids finaux (ton filtre instinctif) : {weights}")
# TEST FINAL :
# RED almost -1 : -0.9942
test_signal = np.array([3.05])
prediction = circuit(weights, test_signal)
print(f"Test signal 3.05 -> Qualia : {prediction:.4f} (Proche de -1 = Rouge)")

Début de la normalisation associative...
Étape 00 - Erreur : 0.0002
Étape 10 - Erreur : 0.0002
Étape 20 - Erreur : 0.0002
Étape 30 - Erreur : 0.0001
Étape 40 - Erreur : 0.0001
Étape 50 - Erreur : 0.0001
Étape 60 - Erreur : 0.0001

Poids finaux (ton filtre instinctif) : [0.0348605]
Test signal 3.05 -> Qualia : -0.9952 (Proche de -1 = Rouge)


In [26]:
print("Début de la normalisation associative...")
for i in range(61):
    weights = opt.step(cost, weights)
    if i % 10 == 0:
        print(f"Étape {i:02d} - Erreur : {cost(weights):.4f}")

print(f"\nPoids finaux (ton filtre instinctif) : {weights}")
# TEST FINAL :
# almost violet 0.0042
test_signal = np.array([1.5665555])
prediction = circuit(weights, test_signal)
print(f"Test signal 1.5665555 -> Qualia : {prediction:.4f} (Proche de 0 = Violet)")

Début de la normalisation associative...
Étape 00 - Erreur : 0.0001
Étape 10 - Erreur : 0.0001
Étape 20 - Erreur : 0.0001
Étape 30 - Erreur : 0.0001
Étape 40 - Erreur : 0.0001
Étape 50 - Erreur : 0.0001
Étape 60 - Erreur : 0.0001

Poids finaux (ton filtre instinctif) : [0.0221998]
Test signal 1.5665555 -> Qualia : 0.0042 (Proche de 0 = Violet)
